# Chapter 10. Spatial Machine Learning with Explainable AI

*Starting With What You Have: A Quantitative Field Guide for Urban Research in Data-Scarce Settings*

Runs in a browser with no installation. Open in Google Colab and choose Runtime, then Run all.


## Step 0. Installation

This is the one chapter with no code-free route. No practical way exists to carry out SHAP analysis through a graphical tool.

In [ ]:
!pip install -q scikit-learn xgboost shap pandas matplotlib

## Step 1. Prepare the data

1,200 observations. The dependent variable is land surface temperature, which is what Chapter 7 produced from satellite imagery.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spatial_ml as sm

df = pd.read_csv("data/urban_heat.csv")
FEATS = ["ndvi", "bld_density", "dist_water_m", "impervious", "elevation_m"]
X = df[FEATS].values
y = df["lst_c"].values
coords = df[["x", "y"]].values
print("observations:", len(df))
df.head()

## Step 2. Check spatial autocorrelation first

**This determines the validation approach in Step 4.** Skip it and every performance figure downstream is inflated.

In [ ]:
from libpysal import weights
from esda.moran import Moran
import geopandas as gpd
from shapely.geometry import Point

g = gpd.GeoDataFrame(geometry=[Point(a, b) for a, b in coords], crs="EPSG:32648")
w = weights.KNN.from_dataframe(g, k=8, use_index=False)
w.transform = "r"
for c in ["ndvi", "bld_density", "lst_c"]:
    print(f"  Moran's I  {c:12s} {Moran(df[c].values, w, permutations=99).I:.3f}")

## Step 3. Fit the models, always with an OLS baseline

These are training-set figures and include overfitting, so they cannot be published.

In [ ]:
from sklearn.metrics import r2_score

models = sm.fit_models(X, y)
for k, mo in models.items():
    print(f"  {k:14s} training R2 = {r2_score(y, mo.predict(X)):.3f}")

## Step 4. Honest validation, the heart of this chapter

Random k-fold splits neighbouring points across training and validation. Where spatial autocorrelation exists, the model sits the exam having already seen the answers.

**The number that goes in the paper is the block figure.** Note also that the degree of inflation differs by model, so selecting on random CV can lead you to the wrong model.

In [ ]:
for k, mo in models.items():
    r = sm.random_cv(mo, X, y)["r2_mean"]
    s = sm.spatial_block_cv(mo, X, y, coords)["r2_mean"]
    print(f"  {k:14s} random {r:.3f} | spatial block {s:.3f} | overstatement {r - s:+.3f}")

## Step 5. SHAP, contribution by variable

Elevation was planted as a pure noise variable. A value near zero for it is evidence the method is working.

In [ ]:
rf = models["RandomForest"]
sv, imp = sm.shap_values(rf, X, FEATS)
imp

## Step 6. Partial dependence, finding the shape of the relationship

A linear model says only that more vegetation means cooler. Partial dependence says **where the effect stops**.

In [ ]:
xs, pdv = sm.partial_dependence(rf, X, FEATS.index("ndvi"))
for v in [0.1, 0.3, 0.45, 0.6, 0.8]:
    i = int(np.argmin(abs(xs - v)))
    print(f"  NDVI {v:.2f} -> predicted LST {pdv[i]:.2f} C")

plt.figure(figsize=(6, 4))
plt.plot(xs, pdv, lw=2.5)
plt.axvline(0.45, ls="--", c="red")
plt.xlabel("NDVI")
plt.ylabel("predicted land surface temperature (C)")
plt.grid(alpha=0.25)
plt.show()

## Step 7. Local SHAP, the link back to Chapter 5

This does the same job as a GWR local coefficient surface, with the difference that no functional form was assumed.

In [ ]:
j = FEATS.index("bld_density")
plt.figure(figsize=(6.5, 5.5))
sc = plt.scatter(coords[:, 0], coords[:, 1], c=sv[:, j], cmap="RdBu_r", s=16,
                 vmin=-abs(sv[:, j]).max(), vmax=abs(sv[:, j]).max())
plt.colorbar(sc, label="SHAP contribution (C)")
plt.xlabel("easting (m)")
plt.ylabel("northing (m)")
plt.show()

d_core = np.hypot(coords[:, 0] - 6000, coords[:, 1] - 6000)
print(f"core, within 3 km : {sv[d_core < 3000, j].mean():+.3f} C")
print(f"periphery, beyond 6 km: {sv[d_core > 6000, j].mean():+.3f} C")

## Step 8. Check what OLS missed

Look at the sign of the impervious surface coefficient. The true planted effect is positive; OLS estimates it as negative.

In [ ]:
from sklearn.linear_model import LinearRegression

ols = LinearRegression().fit(X, y)
for f, c in zip(FEATS, ols.coef_):
    print(f"  {f:14s} {c:+.4f}")

---

**What to do next.** Compare against Section 10.4. SHAP explains what the **model** used to predict, not how the world works. For causal claims, see Chapter 12.